# Phase 7: results table for all eight configurations, confidence intervals and chart

**CRISP-DM stage:** Modelling and evaluation

This is the Phase 5 notebook (`05_results_table`) extended to all eight configurations, with the Phase 7 comparisons in section 4. Nothing else in the calculations changed.

**Differences to keep in mind:** WinCLIP (C7) works at 240 pixels with no fitting, and Reverse Distillation (C8) was trained for 200 epochs with one seed. PatchCore (C4, C5, C6) has 3 seeds, the others 1.

**Runs on:** the laptop, from the results saved by the Kaggle runs. No photos are read and no model runs.

**These are development numbers on half A.** The reported numbers come from the Final test on half B, after every choice is locked.

**Bootstrapping, in plain words:** imagine marking an inspector on only a few vials. If you had picked different vials, would the mark change a lot? The bootstrap pretends to redo the test many times by drawing parts again from the same set, at random, with repeats allowed. The middle 95 percent of the marks is the 95 percent confidence interval. Wide means the number is shaky. It shows uncertainty from having few parts, not whether a method is good.

- **Parts are drawn, not photos,** because the photos of one part under 6 or 7 lights are not independent.
- **Good and defective parts are drawn separately,** keeping their numbers, so every draw has both and AU-ROC can be calculated.
- **2,000 draws with seed 42,** from `src/evaluate.py`.
- **Half A has only 2 good Sheet Metal parts and 3 good Vial parts,** so the intervals are wide and rough.

**Pixel metrics** have no interval here. They are also uncertain because masks were drawn by people, and how much people would disagree cannot be measured here.

## 1. Results table

For PatchCore, each value is the mean over 3 seeds, with the standard deviation (how far seeds typically differ) and the lowest and highest. PaDiM has one seed, so it has one value.

In [ ]:
import csv
import json
import statistics
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

# Lets the notebook run from the project folder or from the notebooks folder
PROJECT_FOLDER = Path.cwd()
if PROJECT_FOLDER.name == "notebooks":
    PROJECT_FOLDER = PROJECT_FOLDER.parent
# Python looks for src in the project folder
sys.path.insert(0, str(PROJECT_FOLDER))
from src import evaluate

CORE_FOLDERS = [PROJECT_FOLDER / "outputs" / "results" / "05_core", PROJECT_FOLDER / "outputs" / "results" / "07_core"]
OUTPUT_FOLDER = PROJECT_FOLDER / "outputs" / "results" / "07_core"


def config_folder(config):
    """Return the folder holding a configuration's results: 05_core for Phase 5 runs, 07_core for Phase 7 runs."""
    for folder in CORE_FOLDERS:
        if (folder / config).exists():
            return folder / config
    raise FileNotFoundError("No results folder for " + config)
FIGURES_FOLDER = PROJECT_FOLDER / "outputs" / "figures"
CONFIGS = ["C1", "C2", "C3", "C4", "C5", "C6", "C7", "C8"]
CONFIG_NAMES = {"C1": "C1 PaDiM, ResNet-18", "C2": "C2 PaDiM, Wide ResNet-50", "C3": "C3 PaDiM, EfficientNet-B0",
                "C4": "C4 PatchCore, ResNet-18", "C5": "C5 PatchCore, Wide ResNet-50", "C6": "C6 PatchCore, EfficientNet-B0",
                "C7": "C7 WinCLIP, CLIP", "C8": "C8 Reverse Distillation, Wide ResNet-50"}
PRODUCTS = ["sheet_metal", "vial"]
METRICS = ["au_pro_5_percent", "au_pro_30_percent", "pixel_f1", "image_f1", "image_au_roc"]
COSTS = ["seconds_per_photo", "peak_gpu_gb", "fit_seconds"]

rows = []
for product in PRODUCTS:
    for config in CONFIGS:
        metric_files = sorted(config_folder(config).glob(config + "_" + product + "_seed*_metrics.json"))
        runs = []
        for metric_file in metric_files:
            with open(metric_file) as file:
                runs.append(json.load(file))

        row = {"product": product, "config": config, "seeds": len(runs)}
        for name in METRICS + COSTS + ["flag_everything_image_f1"]:
            values = []
            for run in runs:
                values.append(float(run[name]))
            row[name + "_mean"] = statistics.mean(values)
            # The standard deviation needs at least 2 seeds
            if len(values) > 1:
                row[name + "_std"] = statistics.stdev(values)
            else:
                row[name + "_std"] = ""
            row[name + "_lowest"] = min(values)
            row[name + "_highest"] = max(values)

        # Each seed's photo scores and threshold, for the bootstrap
        score_lists = []
        thresholds = []
        labels = None
        file_names = None
        for run, metric_file in zip(runs, metric_files):
            scores_file = Path(str(metric_file).replace("_metrics.json", "_photo_scores.csv"))
            scores = []
            these_labels = []
            these_names = []
            with open(scores_file, newline="") as file:
                for photo in csv.DictReader(file):
                    scores.append(float(photo["score"]))
                    these_labels.append(photo["label"] == "bad")
                    these_names.append(photo["file_name"])
            # Every seed must score the same photos in the same order
            if file_names is not None and these_names != file_names:
                raise ValueError("Photo order differs between seeds for " + config + " " + product)
            file_names = these_names
            labels = torch.tensor(these_labels)
            score_lists.append(torch.tensor(scores))
            thresholds.append(float(run["threshold"]))

        intervals = evaluate.bootstrap_image_metrics(score_lists, thresholds, labels, file_names)
        row.update(intervals)
        rows.append(row)

# Numbers only, so this file can be committed
table_file = OUTPUT_FOLDER / "phase7_results_table.csv"
with open(table_file, "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    for row in rows:
        writer.writerow(row)
print("Saved", table_file)

## 2. A readable version of the table

Values are rounded to 3 decimal places. For PatchCore, the standard deviation over seeds is in brackets. Image F1 and image AU-ROC show their 95 percent bootstrap interval.

In [ ]:
def show(row, name):
    """Return a value as text, with its standard deviation over seeds if there is one."""
    text = str(round(row[name + "_mean"], 3))
    if row[name + "_std"] != "":
        text = text + " (" + str(round(row[name + "_std"], 3)) + ")"
    return text


lines = ["# Phase 7 results on half A (development numbers, not final results)", ""]
for product in PRODUCTS:
    lines.append("## " + product)
    lines.append("")
    lines.append("| Configuration | Seeds | AU-PRO 5% | AU-PRO 30% | Pixel F1 | Image F1 [95% interval] | Flag-everything F1 | Image AU-ROC [95% interval] | Seconds per photo | Peak GPU GB |")
    lines.append("| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |")
    for row in rows:
        if row["product"] != product:
            continue
        f1_text = show(row, "image_f1") + " [" + str(round(row["image_f1_low"], 3)) + " to " + str(round(row["image_f1_high"], 3)) + "]"
        auroc_text = show(row, "image_au_roc") + " [" + str(round(row["image_au_roc_low"], 3)) + " to " + str(round(row["image_au_roc_high"], 3)) + "]"
        lines.append("| " + CONFIG_NAMES[row["config"]] + " | " + str(row["seeds"]) + " | " + show(row, "au_pro_5_percent")
                     + " | " + show(row, "au_pro_30_percent") + " | " + show(row, "pixel_f1") + " | " + f1_text
                     + " | " + str(round(row["flag_everything_image_f1_mean"], 3)) + " | " + auroc_text
                     + " | " + show(row, "seconds_per_photo") + " | " + show(row, "peak_gpu_gb") + " |")
    good_parts = rows[[r["product"] for r in rows].index(product)]["good_parts"]
    defective_parts = rows[[r["product"] for r in rows].index(product)]["defective_parts"]
    lines.append("")
    lines.append("Intervals come from drawing " + str(good_parts) + " good and " + str(defective_parts)
                 + " defective parts, 2,000 times. With so few good parts, they are wide and rough.")
    lines.append("")
lines.append("Pixel metrics are uncertain, because masks were drawn by people and their agreement cannot be measured here.")

readable_file = OUTPUT_FOLDER / "phase7_results_table.md"
with open(readable_file, "w", encoding="utf-8") as file:
    for line in lines:
        file.write(line + "\n")
for line in lines:
    print(line)

## 3. Grouped bar chart

- **One panel per product,** because putting 80 bars side by side would be hard to read.
- **Groups:** the 5 headline metrics, all on one scale from 0 to 1.
- **Bars:** one colour per configuration, always in the same order. The colours were checked for colour-blind readers. Two of them are pale against the white background, so the legend and the table above carry the exact values.
- **Error bars:** the lowest to highest over PatchCore's 3 seeds. PaDiM has one seed, so it has none.
- **Dashed lines:** the flag-everything F1, which image F1 must beat.

In [ ]:
# Fixed colours per configuration, checked for colour-blind readers
COLOURS = {"C1": "#2a78d6", "C2": "#eb6834", "C3": "#56b4e9", "C4": "#1baf7a", "C5": "#eda100",
           "C6": "#cc79a7", "C7": "#7a5195", "C8": "#8c6d31"}
METRIC_NAMES = ["AU-PRO 5%", "AU-PRO 30%", "Pixel F1", "Image F1", "Image AU-ROC"]
BAR_WIDTH = 0.1
FIGURE_DPI = 150

figure, axes = plt.subplots(1, 2, figsize=(18, 5.6), sharey=True)
for panel in range(len(PRODUCTS)):
    product = PRODUCTS[panel]
    axis = axes[panel]
    for config_number in range(len(CONFIGS)):
        config = CONFIGS[config_number]
        row = None
        for candidate in rows:
            if candidate["product"] == product and candidate["config"] == config:
                row = candidate
        heights = []
        below = []
        above = []
        for name in METRICS:
            heights.append(row[name + "_mean"])
            below.append(row[name + "_mean"] - row[name + "_lowest"])
            above.append(row[name + "_highest"] - row[name + "_mean"])
        positions = []
        for metric_number in range(len(METRICS)):
            positions.append(metric_number + (config_number - 3.5) * BAR_WIDTH)
        errors = None
        if row["seeds"] > 1:
            errors = [below, above]
        # A thin gap in the background colour separates neighbouring bars
        axis.bar(positions, heights, width=BAR_WIDTH, color=COLOURS[config], label=CONFIG_NAMES[config],
                 edgecolor="#fcfcfb", linewidth=1.5, yerr=errors, capsize=2, error_kw={"elinewidth": 1, "ecolor": "#52514e"})

    # The flag-everything baseline, drawn across the image F1 group only
    baseline = rows[[r["product"] for r in rows].index(product)]["flag_everything_image_f1_mean"]
    group = METRICS.index("image_f1")
    axis.plot([group - 0.45, group + 0.45], [baseline, baseline], linestyle="--", color="#0b0b0b", linewidth=1.2)
    # The label sits above the tallest bar in the group, so bars cannot hide it
    tallest = baseline
    for candidate in rows:
        if candidate["product"] == product and candidate["image_f1_highest"] > tallest:
            tallest = candidate["image_f1_highest"]
    axis.text(group, tallest + 0.03, "dashed: flag everything", ha="center", fontsize=8, color="#52514e")

    axis.set_xticks(range(len(METRICS)))
    axis.set_xticklabels(METRIC_NAMES)
    axis.set_ylim(0, 1)
    axis.set_title(product.replace("_", " ").capitalize() + ", half A", loc="left", fontsize=11)
    axis.grid(axis="y", color="#e4e4e0", linewidth=0.8)
    axis.set_axisbelow(True)
    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
axes[0].set_ylabel("Score (0 to 1)")
# The legend sits below both panels, so it covers no bars
handles, names = axes[0].get_legend_handles_labels()
figure.legend(handles, names, frameon=False, fontsize=9, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.06))
figure.suptitle("Phase 7 development results on half A. Error bars: lowest to highest over 3 PatchCore seeds. Not final results",
                fontsize=10)
figure.tight_layout()
chart_file = FIGURES_FOLDER / "phase7_metric_comparison.png"
figure.savefig(chart_file, dpi=FIGURE_DPI, bbox_inches="tight")
plt.close(figure)
print("Saved", chart_file)

## 4. Testing differences between configurations

To compare two configurations fairly, both are measured on the **same** drawn parts in each draw, and the gap between them is recorded. The middle 95 percent of the 2,000 gaps is the paired interval. **A difference is called real only if this interval leaves out 0.** This is stricter and fairer than checking whether two separate intervals overlap.

- **Method effect:** PatchCore minus PaDiM, with the same network.
- **Network effect:** Wide ResNet-50 minus ResNet-18, with the same method.

AU-PRO and pixel F1 have no intervals, so their differences are not tested here. With only 2 good Sheet Metal parts and 3 good Vial parts in half A, even a real difference rests on very few parts.

In [ ]:
# Far smaller than any real difference in a score from 0 to 1
ROUNDING_NOISE = 1e-9

# Fixed before the Phase 7 results were seen
PAIRS = [
    ("network, PaDiM: EfficientNet-B0 against ResNet-18", "C3", "C1"),
    ("network, PatchCore: EfficientNet-B0 against ResNet-18", "C6", "C4"),
    ("network, PatchCore: EfficientNet-B0 against Wide ResNet-50", "C6", "C5"),
    ("method, Wide ResNet-50: Reverse Distillation against PatchCore", "C8", "C5"),
    ("method: WinCLIP against PatchCore with Wide ResNet-50", "C7", "C5"),
]


def load_scores(config, product):
    """Return each seed's scores and threshold, the labels and the file names."""
    score_lists = []
    thresholds = []
    labels = None
    file_names = None
    for metric_file in sorted(config_folder(config).glob(config + "_" + product + "_seed*_metrics.json")):
        with open(metric_file) as file:
            thresholds.append(float(json.load(file)["threshold"]))
        scores = []
        these_labels = []
        these_names = []
        with open(str(metric_file).replace("_metrics.json", "_photo_scores.csv"), newline="") as file:
            for photo in csv.DictReader(file):
                scores.append(float(photo["score"]))
                these_labels.append(photo["label"] == "bad")
                these_names.append(photo["file_name"])
        score_lists.append(torch.tensor(scores))
        labels = torch.tensor(these_labels)
        file_names = these_names
    return score_lists, thresholds, labels, file_names


difference_rows = []
for product in PRODUCTS:
    for pair_name, first, second in PAIRS:
        first_scores, first_thresholds, labels, first_names = load_scores(first, product)
        second_scores, second_thresholds, second_labels, second_names = load_scores(second, product)
        # Both configurations must have scored the same photos in the same order
        if first_names != second_names or not torch.equal(labels, second_labels):
            raise ValueError("Photos differ between " + first + " and " + second)
        gap = evaluate.paired_bootstrap_difference((first_scores, first_thresholds), (second_scores, second_thresholds),
                                                   labels, first_names)
        gap["product"] = product
        gap["comparison"] = pair_name
        gap["first_minus_second"] = first + " minus " + second
        for metric in ["image_f1", "image_au_roc"]:
            # Rounding in the computer can leave 0.0000000000000001 where the answer is 0,
            # so an edge this close to 0 counts as touching 0
            real = gap[metric + "_difference_low"] > ROUNDING_NOISE or gap[metric + "_difference_high"] < -ROUNDING_NOISE
            gap[metric + "_real"] = real
        difference_rows.append(gap)

# Numbers only, so this file can be committed
differences_file = OUTPUT_FOLDER / "phase7_paired_differences.csv"
with open(differences_file, "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(difference_rows[0].keys()))
    writer.writeheader()
    for row in difference_rows:
        writer.writerow(row)

for row in difference_rows:
    print(row["product"], "|", row["comparison"], "|", row["first_minus_second"],
          "| image F1", round(row["image_f1_difference"], 3),
          "[" + str(round(row["image_f1_difference_low"], 3)) + " to " + str(round(row["image_f1_difference_high"], 3)) + "]",
          "real" if row["image_f1_real"] else "not real",
          "| AU-ROC", round(row["image_au_roc_difference"], 3),
          "[" + str(round(row["image_au_roc_difference_low"], 3)) + " to " + str(round(row["image_au_roc_difference_high"], 3)) + "]",
          "real" if row["image_au_roc_real"] else "not real")
print("Saved", differences_file)